In [0]:
import requests
import json
import boto3

In [0]:
%run ../../helper_function/helper_function

In [0]:
access_key = dbutils.secrets.get(scope="barcakings-secrets",key="s3-access-key")
secret_key = dbutils.secrets.get(scope="barcakings-secrets",key="s3-secret-key")
databricks_token = dbutils.secrets.get(scope="barcakings-secrets",key="dbk-pat")

In [0]:
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
databricks_host  = f"https://{workspace_url}"
print(databricks_host)
headers = {
    "User-Agent": "Mozilla/5.0"
}

In [0]:
dbutils.widgets.text("sport", "football", "sport")
sport = dbutils.widgets.get("sport")

In [0]:
# Read control table
control_df = spark.sql(
    f"""
        SELECT 
            *
        FROM  
            workspace.bk_raw.raw_control_tbl 
        WHERE 
            sport = '{sport}' 
    """)
control_df.display()

In [0]:
control_rows = control_df.collect()

for row in control_rows:
    team = row["team"]
    file_name = row["file_name"]
    tournament = row["tournament"]
    api_url = row["api_url"]
    api_param = json.loads(row["api_param"])
    pn_flag = row["pn_flag"]
    offset = row["offset"]
    limit = row["limit"]
    volume_path = row["target_volume_path"]+file_name+".json"
    bucket_name = row["target_s3_bucket"]
    s3_key = row["target_s3_key"]+file_name+".json"
    top_lvl_filter = row["top_lvl_filter"]

    if pn_flag:
        json_data = pn_api_extraction(api_url,api_param,headers,offset,limit)
    else:
        json_data = api_extraction(api_url,api_param)
        
    upload_s3_vol(
        file_data = json_data, 
        file_format = 'json',
        bucket_name = bucket_name ,
        s3_key = s3_key,
        volume_path = volume_path 
    )

In [0]:
base_path = "/Volumes/workspace/bk_raw/football_raw_files"

control_table_file_name = "raw_control_tbl"

target_s3_bucket = "barca-kings-raw-data"
target_s3_key = "barca-raw-data/control_table/"

temp_path = f"{base_path}/{control_table_file_name}_temp"
final_file = f"{base_path}/{control_table_file_name}.parquet"

control_df.coalesce(1)\
.write.format("parquet")\
.mode("overwrite")\
.option("header", "true")\
.save(temp_path)


# Find generated parquet
files = dbutils.fs.ls(temp_path)

part_file = [
    f.path for f in files
    if f.name.startswith("part-") and f.name.endswith(".parquet")
][0]

# Move and rename
dbutils.fs.mv(part_file, final_file)

# Remove temporary directory
dbutils.fs.rm(temp_path, True)

with open(final_file, "rb" ) as file:
    parquet_file_data = file.read()

#Uploading parquet to S3
upload_s3_vol(
    file_data = parquet_file_data, 
    file_format = 'parquet',
    bucket_name = target_s3_bucket ,
    s3_key = f"{target_s3_key}{control_table_file_name}.parquet",
    volume_path = None 
)

# Remove Volume Files
dbutils.fs.rm(final_file, True)